In [7]:
import pandas as pd 
import json
import os
import datasets as hf_ds
import numpy as np

datasets = {}
ddf = hf_ds.load_dataset("omkarthawakar/FannOrFlop", split="train").to_pandas()

train_df = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/train_data.csv')

# datasets['train'] = train_df

/tmp/ipykernel_1312597/2891334410.py:10: DtypeWarning: Columns (1,2,6,8,9,10,12,13,17,19,23,24,25,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/train_data.csv')


In [31]:
# Convert all columns to strings safely, handling arrays and other complex types
def safe_convert_to_string(x):
    return str(x) 
# Apply safe conversion to all columns
for col in ddf.columns:
    ddf[col] = ddf[col].apply(safe_convert_to_string)

# Ensure column names are strings and reset index
ddf.columns = ddf.columns.astype(str)
ddf = ddf.reset_index(drop=True)

datasets['FannOrFlop'] = ddf

In [32]:
import pandas as pd
import re

def unify_fann_or_flop(df: pd.DataFrame):
    df = df.copy()

    df.rename(columns={
        'source': 'source',
        'title': 'poem_title',
        'tags': 'tags',
        'verse_count': 'verse_count',
        'author': 'poet_name',
        'era': 'poet_era',
        'meter': 'meter',
        'genre': 'genre',
        'id': 'poem_id',
        'explanation': 'overall_explanation',
        'poem_verses': 'poem_text',
        'raw_explanation': 'verses_explanation'
    }, inplace=True)

    def extract_verses(text):
        if not isinstance(text, str):
            return []

        # Split into blocks using verse number markers
        blocks = re.split(r'\n*\b\d+\b\n*', text)
        blocks = [block.strip() for block in blocks if block.strip()]

        verses = []
        for block in blocks:
            lines = [line.strip() for line in block.split('\n') if line.strip()]
            if len(lines) >= 2:
                verses.append('\t'.join(lines))
            elif len(lines) == 1:
                verses.append(lines[0])
        return verses

    def process_tags(tags: str):
        """
        Process the tags string to extract meter, genre, and poet_era.

        Returns:
            tuple: (meter, genre, poet_era) if valid, else None.
        """
        if not isinstance(tags, str):
            return None

        # Remove surrounding brackets if present
        tags = tags.strip().lstrip('[').rstrip(']')

        # Extract values enclosed in single or double quotes
        parts = re.findall(r"'([^']+)'|\"([^\"]+)\"", tags)

        # Flatten the tuple pairs returned by re.findall
        parts = [p1 or p2 for p1, p2 in parts]

        if len(parts) != 3:
            return None

        return tuple(part.strip() for part in parts)

    valid_rows = []
    invalid_rows = []

    for idx, row in df.iterrows():
        verses = extract_verses(row['poem_text'])
        expected_count = pd.to_numeric(row['verse_count'], errors='coerce')

        ### this is commented because after analysis, we found that there are 67 samples with wrong reported verses.
        # # Validate verse count
        # if pd.isna(expected_count) or expected_count != len(verses):
        #     invalid_rows.append(row)
        #     continue

        # Process tags
        sample_tags = str(row.get('tags', ''))
        processed_tags = process_tags(sample_tags)
        if processed_tags is None:
            invalid_rows.append(row)
            continue

        meter, genre, poet_era = processed_tags

        # Update row with processed data
        row['poem_text'] = '\n'.join(verses)
        row['meter'] = meter
        row['genre'] = genre
        row['poet_era'] = poet_era

        valid_rows.append(row)


    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    invalid_df = pd.DataFrame(invalid_rows).reset_index(drop=True)

    print(len(valid_df), "valid rows")
    print(len(invalid_df), "invalid rows")

    # Remove the 'verse_count' column from the dataframe
    valid_df.drop(columns=['verse_count'], inplace=True, errors='ignore')

    return valid_df


In [33]:
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        print(f"Dataset: {key}")
        print("Columns:", dataset.columns.tolist())
        print()
    elif isinstance(dataset, dict):
        print(f"Dataset: {key}")
        print("Keys:", list(dataset.keys()))
        print()
        

Dataset: FannOrFlop
Columns: ['source', 'title', 'tags', 'verse_count', 'author', 'era', 'meter', 'genre', 'id', 'explanation', 'poem_verses', 'raw_explanation']



In [34]:

# Unify all datasets into a single DataFrame
unified_datasets = []

# Iterate through datasets and apply the appropriate unification function
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        if key == 'FannOrFlop':
            df = unify_fann_or_flop(dataset)

        # Add the dataset name as a column
        df['dataset_name'] = key

        # Append the modified DataFrame
        unified_datasets.append(df)

# unified_datasets.append(datasets['train'])

for  dataset in unified_datasets:
    if isinstance(dataset, pd.DataFrame):
        print(f"Dataset: {dataset['dataset_name'].iloc[0]}")
        print("Columns:", dataset.columns.tolist())
        print()
    elif isinstance(dataset, dict):
        print(f"Dataset: {key}")
        print("Keys:", list(dataset.keys()))
        print()
        

6984 valid rows
0 invalid rows
Dataset: FannOrFlop
Columns: ['source', 'poem_title', 'tags', 'poet_name', 'poet_era', 'meter', 'genre', 'poem_id', 'overall_explanation', 'poem_text', 'verses_explanation', 'dataset_name']



In [12]:
# Combine and drop all existing index info

# Ensure all columns are of type str and gather all unique column names
all_columns = set(train_df.columns)
for df in unified_datasets:
    if isinstance(df, pd.DataFrame):
        df.columns = df.columns.astype(str)
        all_columns.update(df.columns)

# Add missing columns with NaN to each dataset
for i, df in enumerate(unified_datasets):
    if isinstance(df, pd.DataFrame):
        missing_columns = all_columns - set(df.columns)
        for col in missing_columns:
            df[col] = pd.NA
        unified_datasets[i] = df[sorted(all_columns)]  # Sort columns for consistency

# Now try the concatenation
all_poems_df = unified_datasets[0]



# Replace "Empty" and NaN values with None
all_poems_df.replace("Empty", None, inplace=True)
# all_poems_df.fillna(value=None, inplace=True)

# Ensure 'poem_verses' column exists and is empty (NA) where missing or blank
if 'poem_verses' not in all_poems_df.columns:
    all_poems_df['poem_verses'] = pd.NA

# Populate 'poem_verses' column if it is empty or NaN
all_poems_df['poem_verses'] = all_poems_df['poem_verses'].fillna(
    all_poems_df['poem_text'].apply(lambda x: x.count('\n') + 1 if isinstance(x, str) else None)
)

# Ensure unique values for the 'poem_id' column
if 'poem_id' in all_poems_df.columns:
    all_poems_df['poem_id'] = all_poems_df.index + 1



/tmp/ipykernel_1312597/1491237601.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_poems_df['poem_verses'] = all_poems_df['poem_verses'].fillna(


In [37]:
import re

def remove_diacritics(text):
    """Remove Arabic diacritics from the given text."""
    arabic_diacritics = re.compile(r'[\u064B-\u0652]')
    return arabic_diacritics.sub('', text)


# Remove rows with empty or NaN values in the 'poem_text' column
all_poems_df = all_poems_df.dropna(subset=['poem_text']).loc[all_poems_df['poem_text'].str.strip() != '']

# Create a new column with poem text without diacritics
all_poems_df['poem_text_no_diacritics'] = all_poems_df['poem_text'].apply(lambda x: remove_diacritics(x) if isinstance(x, str) else x)
# Remove diacritics from the 'poet_name' column
all_poems_df['poet_name_with_diacritics'] = all_poems_df['poet_name']
all_poems_df['poet_name'] = all_poems_df['poet_name'].apply(lambda x: remove_diacritics(x) if isinstance(x, str) else x)


# Filter rows belonging to the 'FannOrFlop' dataset
test_df = all_poems_df[all_poems_df['dataset_name'] == 'FannOrFlop']
# train_df = all_poems_df[all_poems_df['dataset_name'] != 'FannOrFlop']


# Count empty, NaN, None, or 'Empty' values **per column**
empty_nan_none_counts = {
    column: (
        all_poems_df[column].apply(
            lambda x: pd.isna(x) or x == '' or x == 'Empty'
        ).sum()
    )
    for column in all_poems_df.columns
}

# Display the counts
for col, count in empty_nan_none_counts.items():
    print(f"{col}: {count}")

print('Dedupe test data shape:', test_df.shape)

corrupted_poem: 6984
corruption_assigned_template: 6984
corruption_metadata: 6984
corruption_type: 6984
dataset_name: 0
genre: 0
key_phrases: 6984
keywords: 6984
location: 6984
meter: 0
overall_explanation: 0
poem_id: 0
poem_language: 6984
poem_text: 0
poem_text_no_diacritics: 0
poem_title: 0
poem_type: 6984
poem_url: 6984
poem_verses: 0
poet_description: 6984
poet_english_id: 6984
poet_era: 0
poet_id: 6984
poet_name: 0
poet_name_with_diacritics: 0
poet_url: 6984
rhyme: 6984
source: 0
tags: 0
verses_explanation: 0
Dedupe test data shape: (6984, 30)


In [36]:
test_df

,corrupted_poem,corruption_assigned_template,corruption_metadata,corruption_type,dataset_name,genre,key_phrases,keywords,location,meter,...,poet_english_id,poet_era,poet_id,poet_name,poet_name_with_diacritics,poet_url,rhyme,source,tags,verses_explanation
0,<NA>,<NA>,<NA>,<NA>,FannOrFlop,سياسية,<NA>,<NA>,<NA>,البسيط,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,<NA>,https://arabic-poetry.net/poem/20236-حياكم-الل...,['البسيط' 'سياسية' 'العصر الحديث'],Verses 1-12:\nبسم الله الرحمن الرحيم،\n\nهذه ق...
1,<NA>,<NA>,<NA>,<NA>,FannOrFlop,رثاء,<NA>,<NA>,<NA>,الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,<NA>,https://arabic-poetry.net/poem/20345-غاب-الأدي...,['الكامل' 'رثاء' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة رثاء في أديب مصري، يم...
2,<NA>,<NA>,<NA>,<NA>,FannOrFlop,مدح,<NA>,<NA>,<NA>,الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,<NA>,https://arabic-poetry.net/poem/20152-عثمان-إنك...,['الكامل' 'مدح' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة مدحٌ لعُثمان بن عفان ...
3,<NA>,<NA>,<NA>,<NA>,FannOrFlop,عتاب,<NA>,<NA>,<NA>,الخفيف,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,<NA>,https://arabic-poetry.net/poem/20188-إن-عضيك-ي...,['الخفيف' 'عتاب' 'العصر الحديث'],Verses 1-10:\nتتحدث هذه القصيدة عن لوعة الشاعر...
4,<NA>,<NA>,<NA>,<NA>,FannOrFlop,عتاب,<NA>,<NA>,<NA>,الرجز,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,<NA>,https://arabic-poetry.net/poem/20185-من-واجد-م...,['الرجز' 'عتاب' 'العصر الحديث'],Verses 1-12:\nتتحدث هذه القصيدة عن شاعرٍ مُتعب...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,<NA>,<NA>,<NA>,<NA>,FannOrFlop,صبر,<NA>,<NA>,<NA>,البسيط,...,<NA>,العصر العباسي,<NA>,البحتري,البحتري,<NA>,<NA>,https://arabic-poetry.net/poem/4547-إليك-ما-أن...,['البسيط' 'صبر' 'العصر العباسي'],Verses 1-12:\nهذه القصيدة تعبر عن شاعر يواجه ص...
6980,<NA>,<NA>,<NA>,<NA>,FannOrFlop,غزل,<NA>,<NA>,<NA>,الوافر,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,<NA>,https://arabic-poetry.net/poem/4474-خذ-العيش-ا...,['الوافر' 'غزل' 'العصر العباسي'],Verses 1-12:\nتتحدث هذه القصيدة عن متعة الشراب...
6981,<NA>,<NA>,<NA>,<NA>,FannOrFlop,صبر,<NA>,<NA>,<NA>,الكامل,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,<NA>,https://arabic-poetry.net/poem/3786-اصبر-لمر-ح...,['الكامل' 'صبر' 'العصر العباسي'],Verses 1-11:\nتتحدث هذه القصيدة عن الموت والحس...
6982,<NA>,<NA>,<NA>,<NA>,FannOrFlop,سياسية,<NA>,<NA>,<NA>,الطويل,...,<NA>,العصر العباسي,<NA>,أبو العتاهية,أبو العتاهية,<NA>,<NA>,https://arabic-poetry.net/poem/3146-لخير-إمام-...,['الطويل' 'سياسية' 'العصر العباسي'],Verses 1-10:\nهذه قصيدة تُعبر عن شكوى وتضرع إل...


In [59]:

# 1. Merge `gnre` into `genre` if genre is missing or empty
for df in [test_df]:
    df['genre'] = df['genre'].fillna('').astype(str).str.strip()

# 2. Strip leading/trailing spaces from all string columns
def strip_all_columns(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()
    return df

test_df = strip_all_columns(test_df)

# 3. Define mappings
country_names = [
    'مصر', 'لبنان', 'المغرب', 'سوريا', 'تونس', 'فلسطين', 'موريتانيا', 'العراق',
    'اليمن', 'ليبيا', 'السعودية', 'السودان', 'الجزائر', 'الأردن', 'عمان',
    'الإمارات',
]

era_mapping = {
    'الحديث': 'العصر الحديث',
    'العثماني': 'العصر العثماني',
    'المملوكي': 'العصر المملوكي',
    'العباسي': 'العصر العباسي',
    'الأموي': 'العصر الأموي',
    'الفاطمي': 'العصر الفاطمي',
    'الأيوبي': 'العصر الأيوبي',
    'قبل الإسلام': 'العصر الجاهلي',
    'العصر الايوبي': 'العصر الأيوبي',
    'العصر الاموي': 'العصر الأموي',
    'الدولة الايوبية': 'العصر الأيوبي',
    'الدولة الفاطمية': 'العصر الفاطمي',
    'الدولة المملوكية': 'العصر المملوكي',
    'بين الدولتين' : 'عصر بين الدولتين',
    'الإسلامي': 'العصر الإسلامي',
    'العصر الاسلامي': 'العصر الإسلامي',
    'قبل الإسلام': 'العصر الجاهلي',
    
    # Standardize all to المخضرمون
    'المخضرمون': 'المخضرمون',
    'المخضرمين': 'المخضرمون',
    'الشعراء المخضرمون': 'المخضرمون',
}


def normalize_poet_era_location_and_meter(df):
    for idx, row in df.iterrows():
        # Normalize poet_era and location
        era = row.get('poet_era', '')
        location = row.get('location', '')
        
        if era in country_names:
            if not location or location.strip() == '':
                df.at[idx, 'location'] = era
            elif era not in location:
                df.at[idx, 'location'] = era  # Overwrite to keep clean
            df.at[idx, 'poet_era'] = 'غير محدد'
        elif era in era_mapping:
            df.at[idx, 'poet_era'] = era_mapping[era]
        
        # Normalize meter
        meter = row.get('meter', '')
        if meter:
            # Remove surrounding whitespace
            meter = meter.strip()
            
            # If meter starts with "البحر" or does not start with "بحر"
            if meter.startswith("البحر"):
                normalized = re.sub(r"^البحر\s+", "", meter)
                df.at[idx, 'meter'] = f"بحر {normalized}"
            elif not meter.startswith("بحر"):
                df.at[idx, 'meter'] = f"بحر {meter}"



    # Set 'poet_era' to None if its value is in the specified list
    df['poet_era'] = df['poet_era'].apply(lambda x: None if x in ["عصرين", "غير محدد"] else x)
    
    # Normalize 'meter' column for specific cases
    df['meter'] = df['meter'].apply(lambda x: None if x in ["بحر nan", "بحر None", "بحر"] else x)
    return df


def detect_rhyme(poem_text):
    """
    Detect rhyme character from poem verses.
    If at least 70% of verses end with the same character, return that character as rhyme.
    """
    if pd.isna(poem_text) or not isinstance(poem_text, str):
        return None
    
    # Split poem into verses
    verses = [verse.strip() for verse in poem_text.split('\n') if verse.strip()]
    
    if len(verses) < 2:  # Need at least 2 verses to determine rhyme
        return None
    
    # Count ending characters
    ending_chars = {}
    for verse in verses:
        if verse:  # Make sure verse is not empty
            last_char = verse[-1]
            ending_chars[last_char] = ending_chars.get(last_char, 0) + 1
    
    # Find the most common ending character
    if not ending_chars:
        return None
    
    max_count = max(ending_chars.values())
    total_verses = len(verses)
    
    # Check if at least 70% of verses end with the same character
    if max_count / total_verses >= 0.7:
        # Return the character that appears most frequently
        for char, count in ending_chars.items():
            if count == max_count:
                return char
    
    return pd.NA


if 'rhyme' not in test_df.columns:
    test_df['rhyme'] = pd.NA
    
# Apply to both splits
test_df = normalize_poet_era_location_and_meter(test_df)
    

# capture pd.NA, empty string, or the literal "<NA>"
mask = test_df['rhyme'].isna() | (test_df['rhyme'].isin(['', '<NA>']))
test_df.loc[mask, 'rhyme'] = test_df.loc[mask, 'poem_text_no_diacritics'].apply(detect_rhyme)




In [60]:
test_df

,corrupted_poem,corruption_assigned_template,corruption_metadata,corruption_type,dataset_name,genre,key_phrases,keywords,location,meter,...,poet_english_id,poet_era,poet_id,poet_name,poet_name_with_diacritics,poet_url,rhyme,source,tags,verses_explanation
0,<NA>,<NA>,<NA>,<NA>,FannOrFlop,سياسية,<NA>,<NA>,<NA>,بحر البسيط,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ا,https://arabic-poetry.net/poem/20236-حياكم-الل...,['البسيط' 'سياسية' 'العصر الحديث'],Verses 1-12:\nبسم الله الرحمن الرحيم،\n\nهذه ق...
1,<NA>,<NA>,<NA>,<NA>,FannOrFlop,رثاء,<NA>,<NA>,<NA>,بحر الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ا,https://arabic-poetry.net/poem/20345-غاب-الأدي...,['الكامل' 'رثاء' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة رثاء في أديب مصري، يم...
2,<NA>,<NA>,<NA>,<NA>,FannOrFlop,مدح,<NA>,<NA>,<NA>,بحر الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ل,https://arabic-poetry.net/poem/20152-عثمان-إنك...,['الكامل' 'مدح' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة مدحٌ لعُثمان بن عفان ...
3,<NA>,<NA>,<NA>,<NA>,FannOrFlop,عتاب,<NA>,<NA>,<NA>,بحر الخفيف,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,م,https://arabic-poetry.net/poem/20188-إن-عضيك-ي...,['الخفيف' 'عتاب' 'العصر الحديث'],Verses 1-10:\nتتحدث هذه القصيدة عن لوعة الشاعر...
4,<NA>,<NA>,<NA>,<NA>,FannOrFlop,عتاب,<NA>,<NA>,<NA>,بحر الرجز,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,م,https://arabic-poetry.net/poem/20185-من-واجد-م...,['الرجز' 'عتاب' 'العصر الحديث'],Verses 1-12:\nتتحدث هذه القصيدة عن شاعرٍ مُتعب...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,<NA>,<NA>,<NA>,<NA>,FannOrFlop,صبر,<NA>,<NA>,<NA>,بحر البسيط,...,<NA>,العصر العباسي,<NA>,البحتري,البحتري,<NA>,ب,https://arabic-poetry.net/poem/4547-إليك-ما-أن...,['البسيط' 'صبر' 'العصر العباسي'],Verses 1-12:\nهذه القصيدة تعبر عن شاعر يواجه ص...
6980,<NA>,<NA>,<NA>,<NA>,FannOrFlop,غزل,<NA>,<NA>,<NA>,بحر الوافر,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,س,https://arabic-poetry.net/poem/4474-خذ-العيش-ا...,['الوافر' 'غزل' 'العصر العباسي'],Verses 1-12:\nتتحدث هذه القصيدة عن متعة الشراب...
6981,<NA>,<NA>,<NA>,<NA>,FannOrFlop,صبر,<NA>,<NA>,<NA>,بحر الكامل,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,<NA>,https://arabic-poetry.net/poem/3786-اصبر-لمر-ح...,['الكامل' 'صبر' 'العصر العباسي'],Verses 1-11:\nتتحدث هذه القصيدة عن الموت والحس...
6982,<NA>,<NA>,<NA>,<NA>,FannOrFlop,سياسية,<NA>,<NA>,<NA>,بحر الطويل,...,<NA>,العصر العباسي,<NA>,أبو العتاهية,أبو العتاهية,<NA>,ر,https://arabic-poetry.net/poem/3146-لخير-إمام-...,['الطويل' 'سياسية' 'العصر العباسي'],Verses 1-10:\nهذه قصيدة تُعبر عن شكوى وتضرع إل...


In [61]:
test_6917 = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/poetry_6917.csv')

# Match poems using poem_text_no_diacritics and update specific columns in all_test
columns_to_update = ['keywords', 'key_phrases', 'corrupted_poem', 'corruption_assigned_template', 'corruption_type', 'corruption_metadata']

# Create a copy of all_test to update
all_test_updated = test_df.copy()

# Create a mapping from poem_text_no_diacritics to the columns we want to update from test_6917
update_mapping = test_6917.set_index('poem_text_no_diacritics')[columns_to_update].to_dict('index')

print(f"Number of poems to match: {len(update_mapping)}")
print(f"Columns to update: {columns_to_update}")

# Check which poems in all_test have matching poem_text_no_diacritics in test_6917
matching_mask = all_test_updated['poem_text_no_diacritics'].isin(test_6917['poem_text_no_diacritics'])
print(f"Number of matching poems in all_test: {matching_mask.sum()}")

# Update the columns for matching rows
for idx, row in all_test_updated.iterrows():
    poem_text = row['poem_text_no_diacritics']
    if poem_text in update_mapping:
        for col in columns_to_update:
            all_test_updated.at[idx, col] = update_mapping[poem_text][col]

Number of poems to match: 6917
Columns to update: ['keywords', 'key_phrases', 'corrupted_poem', 'corruption_assigned_template', 'corruption_type', 'corruption_metadata']
Number of matching poems in all_test: 6917


In [62]:
def is_empty_value(x):
    # pd.isna covers np.nan, pd.NA, None
    if pd.isna(x):
        return True
    # Some objects (lists, dicts, arrays) - consider empty containers as empty
    try:
        if hasattr(x, '__len__') and not isinstance(x, (str, bytes)) and len(x) == 0:
            return True
    except Exception:
        pass
    # Normalize to string and check common empty-like representations
    s = str(x).strip()
    if s == '' or s.lower() in {'empty', 'nan', 'none', '<na>'} or s in {'[]', '[ ]', '{}'}:
        return True
    return False

empty_nan_none_counts = {
    col: all_test_updated[col].apply(is_empty_value).sum()
    for col in all_test_updated.columns
}

for col, count in empty_nan_none_counts.items():
    print(f"{col}: {count}")

corrupted_poem: 3121
corruption_assigned_template: 3120
corruption_metadata: 3120
corruption_type: 3120
dataset_name: 0
genre: 0
key_phrases: 3123
keywords: 3123
location: 6984
meter: 0
overall_explanation: 0
poem_id: 0
poem_language: 6984
poem_text: 0
poem_text_no_diacritics: 0
poem_title: 0
poem_type: 6984
poem_url: 6984
poem_verses: 0
poet_description: 6984
poet_english_id: 6984
poet_era: 0
poet_id: 6984
poet_name: 0
poet_name_with_diacritics: 0
poet_url: 6984
rhyme: 669
source: 0
tags: 0
verses_explanation: 0


In [63]:
all_test_updated

,corrupted_poem,corruption_assigned_template,corruption_metadata,corruption_type,dataset_name,genre,key_phrases,keywords,location,meter,...,poet_english_id,poet_era,poet_id,poet_name,poet_name_with_diacritics,poet_url,rhyme,source,tags,verses_explanation
0,NaN,NaN,NaN,NaN,FannOrFlop,سياسية,NaN,NaN,<NA>,بحر البسيط,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ا,https://arabic-poetry.net/poem/20236-حياكم-الل...,['البسيط' 'سياسية' 'العصر الحديث'],Verses 1-12:\nبسم الله الرحمن الرحيم،\n\nهذه ق...
1,NaN,NaN,NaN,NaN,FannOrFlop,رثاء,NaN,NaN,<NA>,بحر الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ا,https://arabic-poetry.net/poem/20345-غاب-الأدي...,['الكامل' 'رثاء' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة رثاء في أديب مصري، يم...
2,NaN,NaN,NaN,NaN,FannOrFlop,مدح,NaN,NaN,<NA>,بحر الكامل,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,ل,https://arabic-poetry.net/poem/20152-عثمان-إنك...,['الكامل' 'مدح' 'العصر الحديث'],Verses 1-3:\nهذه القصيدة مدحٌ لعُثمان بن عفان ...
3,يا أخي، عَضُّكَ إِيَّايَ بالمَلامِ لا يُؤَدّي ...,6.0,"poem_text, poet_era, poet_name, meter",meter_destruction,FannOrFlop,عتاب,"[""غير راعي الذمام"", ""خشيت رد السلام"", ""تمنى لو...","[""الخيانة"", ""الألم"", ""الغربة""]",<NA>,بحر الخفيف,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,م,https://arabic-poetry.net/poem/20188-إن-عضيك-ي...,['الخفيف' 'عتاب' 'العصر الحديث'],Verses 1-10:\nتتحدث هذه القصيدة عن لوعة الشاعر...
4,مِن واجِدٍ مُنَقِّرِ مَضجَعِهِ طَريدَ دَهرٍ جا...,1.0,"poem_text, rhyme",rhyme_structure,FannOrFlop,عتاب,"[""طريد دهر جائر الأحكام"", ""يا نزهة الأنام"", ""أ...","[""المعاناة"", ""الشوق"", ""الفناء""]",<NA>,بحر الرجز,...,<NA>,العصر الحديث,<NA>,حافظ ابراهيم,حافظ ابراهيم,<NA>,م,https://arabic-poetry.net/poem/20185-من-واجد-م...,['الرجز' 'عتاب' 'العصر الحديث'],Verses 1-12:\nتتحدث هذه القصيدة عن شاعرٍ مُتعب...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,NaN,NaN,NaN,NaN,FannOrFlop,صبر,NaN,NaN,<NA>,بحر البسيط,...,<NA>,العصر العباسي,<NA>,البحتري,البحتري,<NA>,ب,https://arabic-poetry.net/poem/4547-إليك-ما-أن...,['البسيط' 'صبر' 'العصر العباسي'],Verses 1-12:\nهذه القصيدة تعبر عن شاعر يواجه ص...
6980,خُذِ العَيشَ مِنَ المَجوسِ هَنِيّاً مُعَاقِر...,7.0,"poem_text, meter, poet_era, poet_name",meter_inconsistency,FannOrFlop,غزل,"[""خذ العيش الهني"", ""ودع لهو العريب بطرد صيد"", ...","[""المتعة"", ""الخمر"", ""العيش الهني""]",<NA>,بحر الوافر,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,س,https://arabic-poetry.net/poem/4474-خذ-العيش-ا...,['الوافر' 'غزل' 'العصر العباسي'],Verses 1-12:\nتتحدث هذه القصيدة عن متعة الشراب...
6981,اصْبِرْ عَلى ما جاءَ مِنْ دَهْرِ\tفَسَتَحْمَدَ...,5.0,"poem_text, poet_era, target_meter, poet_name, ...",meter_transformation,FannOrFlop,صبر,"[""اصـبر لمـر حـوادث الـدهر"", ""يـا ليت شعري كيف...","[""الموت"", ""الآخرة"", ""الندم""]",<NA>,بحر الكامل,...,<NA>,العصر العباسي,<NA>,أبو نواس,أبو نواس,<NA>,<NA>,https://arabic-poetry.net/poem/3786-اصبر-لمر-ح...,['الكامل' 'صبر' 'العصر العباسي'],Verses 1-11:\nتتحدث هذه القصيدة عن الموت والحس...
6982,لِشَرِّ خروفٍ نامَ في خيرِ مَعصَرِ\tوَأَبخَلِ ...,2.0,"poem_text, rhyme",rhyme_content,FannOrFlop,سياسية,"[""كتبت وعيني تستهل دمعها"", ""أنهب أموالي وخرب أ...","[""الظلم"", ""الاستغاثة"", ""القرابة""]",<NA>,بحر الطويل,...,<NA>,العصر العباسي,<NA>,أبو العتاهية,أبو العتاهية,<NA>,ر,https://arabic-poetry.net/poem/3146-لخير-إمام-...,['الطويل' 'سياسية' 'العصر العباسي'],Verses 1-10:\nهذه قصيدة تُعبر عن شكوى وتضرع إل...


In [64]:
all_test_updated.to_csv('/path/to/data/test_data_v2.csv', index=False)

print("✅ Cleaned data saved to data/cleaned_train.tsv and data/cleaned_test.tsv")



total_count = 0
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        count = len(dataset)
        print(f"{key}: {count} rows")
        total_count += count
    elif isinstance(dataset, dict):
        count = len(dataset)
        print(f"{key}: {count} entries")
        total_count += count

print(f"Total: {total_count}")


✅ Cleaned data saved to data/cleaned_train.tsv and data/cleaned_test.tsv
FannOrFlop: 6984 rows
Total: 6984


In [69]:
# Compare the two test datasets
import pandas as pd

# Load both datasets
df1 = pd.read_csv('/path/to/data/test_data_v2.csv')
df2 = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/poetry_6917.csv')



In [77]:
df1['poet_name']

0       حافظ ابراهيم
1       حافظ ابراهيم
2       حافظ ابراهيم
3       حافظ ابراهيم
4       حافظ ابراهيم
            ...     
6979         البحتري
6980        أبو نواس
6981        أبو نواس
6982    أبو العتاهية
6983         البحتري
Name: poet_name, Length: 6984, dtype: object

In [70]:
# First, let's explore the structure of both DataFrames
print("DF1 Shape:", df1.shape)
print("DF2 Shape:", df2.shape)
print("\nDF1 Columns:", list(df1.columns))
print("\nDF2 Columns:", list(df2.columns))

# Check if both have poem_text column
print(f"\nDF1 has 'poem_text': {'poem_text' in df1.columns}")
print(f"DF2 has 'poem_text': {'poem_text' in df2.columns}")

# Show sample data
print("\nDF1 first few rows:")
print(df1.head(2))
print("\nDF2 first few rows:")
print(df2.head(2))

DF1 Shape: (6984, 30)
DF2 Shape: (6917, 29)

DF1 Columns: ['corrupted_poem', 'corruption_assigned_template', 'corruption_metadata', 'corruption_type', 'dataset_name', 'genre', 'key_phrases', 'keywords', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text', 'poem_text_no_diacritics', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_name_with_diacritics', 'poet_url', 'rhyme', 'source', 'tags', 'verses_explanation']

DF2 Columns: ['dataset_name', 'genre', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_url', 'rhyme', 'source', 'verses_explanation', 'poem_text_no_diacritics', 'poet_name_with_diacritics', 'keywords', 'key_phrases', 'corrupted_poem', 'corruption_assigned_template', 'corruption_type', 'corrup

In [71]:
# Match the two DataFrames based on poem_text column
# First, let's check for exact matches
matched_df1 = df1[df1['poem_text'].isin(df2['poem_text'])]
matched_df2 = df2[df2['poem_text'].isin(df1['poem_text'])]

print(f"Poems in DF1 that match DF2: {len(matched_df1)}")
print(f"Poems in DF2 that match DF1: {len(matched_df2)}")

# Let's also check for duplicates within each dataset
print(f"Duplicate poem_text in DF1: {df1['poem_text'].duplicated().sum()}")
print(f"Duplicate poem_text in DF2: {df2['poem_text'].duplicated().sum()}")

# Find common columns between the two DataFrames (excluding poem_text)
common_cols = set(df1.columns).intersection(set(df2.columns))
common_cols.discard('poem_text')  # We'll use this as the key
print(f"\nCommon columns for comparison: {sorted(common_cols)}")

Poems in DF1 that match DF2: 6917
Poems in DF2 that match DF1: 6917
Duplicate poem_text in DF1: 0
Duplicate poem_text in DF2: 0

Common columns for comparison: ['corrupted_poem', 'corruption_assigned_template', 'corruption_metadata', 'corruption_type', 'dataset_name', 'genre', 'key_phrases', 'keywords', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text_no_diacritics', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_name_with_diacritics', 'poet_url', 'rhyme', 'source', 'verses_explanation']


In [72]:
# Merge the DataFrames on poem_text to compare values
merged_df = pd.merge(
    df1[df1['poem_text'].isin(df2['poem_text'])], 
    df2, 
    on='poem_text', 
    suffixes=('_df1', '_df2'),
    how='inner'
)

print(f"Successfully merged {len(merged_df)} rows")

# Function to safely compare values (handles NaN values)
def safe_compare(val1, val2):
    if pd.isna(val1) and pd.isna(val2):
        return True
    elif pd.isna(val1) or pd.isna(val2):
        return False
    else:
        return str(val1).strip() == str(val2).strip()

# Compare each common column
differences = {}
for col in sorted(common_cols):
    col1 = f"{col}_df1"
    col2 = f"{col}_df2"
    
    if col1 in merged_df.columns and col2 in merged_df.columns:
        # Compare values
        matches = [safe_compare(v1, v2) for v1, v2 in zip(merged_df[col1], merged_df[col2])]
        num_matches = sum(matches)
        num_differences = len(matches) - num_matches
        
        if num_differences > 0:
            differences[col] = {
                'matches': num_matches,
                'differences': num_differences,
                'percentage_match': (num_matches / len(matches)) * 100
            }

# Display results
if differences:
    print("\nColumns with differences:")
    for col, stats in differences.items():
        print(f"{col}: {stats['differences']} differences out of {stats['differences'] + stats['matches']} rows ({stats['percentage_match']:.2f}% match)")
else:
    print("\n✅ All common columns have identical values!")

print(f"\nTotal columns compared: {len(common_cols)}")
print(f"Columns with differences: {len(differences)}")
print(f"Columns with identical values: {len(common_cols) - len(differences)}")

Successfully merged 6917 rows

Columns with differences:
poem_id: 6907 differences out of 6917 rows (0.14% match)
poet_name_with_diacritics: 4620 differences out of 6917 rows (33.21% match)

Total columns compared: 28
Columns with differences: 2
Columns with identical values: 26


In [73]:
# Let's examine the specific differences in detail

# 1. Check poem_id differences
print("=== POEM_ID DIFFERENCES ===")
poem_id_diff = merged_df[~merged_df.apply(lambda row: safe_compare(row['poem_id_df1'], row['poem_id_df2']), axis=1)]
print(f"Number of rows with poem_id differences: {len(poem_id_diff)}")
if len(poem_id_diff) > 0:
    print("Sample differences:")
    sample_size = min(5, len(poem_id_diff))
    for i in range(sample_size):
        row = poem_id_diff.iloc[i]
        print(f"  Row {i+1}: DF1={row['poem_id_df1']}, DF2={row['poem_id_df2']}")

print("\n" + "="*50)

# 2. Check poet_name_with_diacritics differences
print("=== POET_NAME_WITH_DIACRITICS DIFFERENCES ===")
poet_name_diff = merged_df[~merged_df.apply(lambda row: safe_compare(row['poet_name_with_diacritics_df1'], row['poet_name_with_diacritics_df2']), axis=1)]
print(f"Number of rows with poet_name_with_diacritics differences: {len(poet_name_diff)}")
if len(poet_name_diff) > 0:
    print("Sample differences:")
    sample_size = min(5, len(poet_name_diff))
    for i in range(sample_size):
        row = poet_name_diff.iloc[i]
        print(f"  Row {i+1}: DF1='{row['poet_name_with_diacritics_df1']}', DF2='{row['poet_name_with_diacritics_df2']}'")

print("\n" + "="*50)

=== POEM_ID DIFFERENCES ===
Number of rows with poem_id differences: 6907
Sample differences:
  Row 1: DF1=12, DF2=11
  Row 2: DF1=13, DF2=12
  Row 3: DF1=14, DF2=13
  Row 4: DF1=15, DF2=14
  Row 5: DF1=16, DF2=15

=== POET_NAME_WITH_DIACRITICS DIFFERENCES ===
Number of rows with poet_name_with_diacritics differences: 4620
Sample differences:
  Row 1: DF1='طرفة بن العبد', DF2='طَرَفَةُ بنُ العبْدِ'
  Row 2: DF1='طرفة بن العبد', DF2='طَرَفَةُ بنُ العبْدِ'
  Row 3: DF1='طرفة بن العبد', DF2='طَرَفَةُ بنُ العبْدِ'
  Row 4: DF1='طرفة بن العبد', DF2='طَرَفَةُ بنُ العبْدِ'
  Row 5: DF1='طرفة بن العبد', DF2='طَرَفَةُ بنُ العبْدِ'



In [74]:
# Create a comprehensive comparison summary
print("=== COMPREHENSIVE COMPARISON SUMMARY ===")
print(f"Total poems in DF1: {len(df1):,}")
print(f"Total poems in DF2: {len(df2):,}")
print(f"Poems that match on poem_text: {len(merged_df):,}")
print(f"Additional poems in DF1: {len(df1) - len(merged_df):,}")

print(f"\nCommon columns compared: {len(common_cols)}")
print(f"Columns with identical values: {len(common_cols) - len(differences)}")
print(f"Columns with differences: {len(differences)}")

print("\n=== ANALYSIS OF DIFFERENCES ===")

if 'poem_id' in differences:
    print("📊 POEM_ID differences:")
    print("   - This appears to be a sequential ID offset")
    print("   - DF1 IDs seem to start from a higher number than DF2")
    print("   - This is likely due to different indexing/ID assignment methods")

if 'poet_name_with_diacritics' in differences:
    print("📊 POET_NAME_WITH_DIACRITICS differences:")
    print("   - DF1 has names without diacritics")
    print("   - DF2 has names with Arabic diacritics")
    print("   - This is a formatting difference, not content difference")

print("\n=== CONCLUSION ===")
if len(differences) <= 2 and all(col in ['poem_id', 'poet_name_with_diacritics'] for col in differences.keys()):
    print("✅ The DataFrames are essentially identical!")
    print("   - All content columns match perfectly")
    print("   - Only differences are in ID formatting and diacritics")
    print("   - These are expected technical differences, not data discrepancies")
else:
    print("⚠️  There are significant differences between the DataFrames")

# Check if DF1 has any poems not in DF2
df1_only = df1[~df1['poem_text'].isin(df2['poem_text'])]
if len(df1_only) > 0:
    print(f"\n📋 DF1 has {len(df1_only)} additional poems not found in DF2")
else:
    print(f"\n📋 All poems in DF1 are also in DF2")

=== COMPREHENSIVE COMPARISON SUMMARY ===
Total poems in DF1: 6,984
Total poems in DF2: 6,917
Poems that match on poem_text: 6,917
Additional poems in DF1: 67

Common columns compared: 28
Columns with identical values: 26
Columns with differences: 2

=== ANALYSIS OF DIFFERENCES ===
📊 POEM_ID differences:
   - This appears to be a sequential ID offset
   - DF1 IDs seem to start from a higher number than DF2
   - This is likely due to different indexing/ID assignment methods
📊 POET_NAME_WITH_DIACRITICS differences:
   - DF1 has names without diacritics
   - DF2 has names with Arabic diacritics
   - This is a formatting difference, not content difference

=== CONCLUSION ===
✅ The DataFrames are essentially identical!
   - All content columns match perfectly
   - Only differences are in ID formatting and diacritics
   - These are expected technical differences, not data discrepancies

📋 DF1 has 67 additional poems not found in DF2


In [75]:
# Show detailed breakdown of which columns match perfectly
perfectly_matched_cols = []
for col in sorted(common_cols):
    if col not in differences:
        perfectly_matched_cols.append(col)

print("=== PERFECTLY MATCHED COLUMNS ===")
print("The following columns have 100% identical values:")
for i, col in enumerate(perfectly_matched_cols, 1):
    print(f"{i:2d}. {col}")

print(f"\nTotal perfectly matched columns: {len(perfectly_matched_cols)}")

print("\n=== COLUMNS WITH DIFFERENCES ===")
for col, stats in differences.items():
    print(f"- {col}: {stats['percentage_match']:.2f}% match ({stats['differences']} differences)")

# Optional: Save the comparison results
comparison_results = {
    'total_df1_rows': len(df1),
    'total_df2_rows': len(df2),
    'matched_rows': len(merged_df),
    'additional_in_df1': len(df1) - len(merged_df),
    'perfectly_matched_columns': perfectly_matched_cols,
    'columns_with_differences': differences
}

print(f"\n=== FINAL VERDICT ===")
print("✅ SUCCESS: The two DataFrames are substantially identical!")
print("💡 Key findings:")
print("   • All 6,917 poems from DF2 are present in DF1")
print("   • 26 out of 28 common columns have identical values")
print("   • Only differences are technical (IDs and diacritics)")
print("   • DF1 contains 67 additional poems not in DF2")

=== PERFECTLY MATCHED COLUMNS ===
The following columns have 100% identical values:
 1. corrupted_poem
 2. corruption_assigned_template
 3. corruption_metadata
 4. corruption_type
 5. dataset_name
 6. genre
 7. key_phrases
 8. keywords
 9. location
10. meter
11. overall_explanation
12. poem_language
13. poem_text_no_diacritics
14. poem_title
15. poem_type
16. poem_url
17. poem_verses
18. poet_description
19. poet_english_id
20. poet_era
21. poet_id
22. poet_name
23. poet_url
24. rhyme
25. source
26. verses_explanation

Total perfectly matched columns: 26

=== COLUMNS WITH DIFFERENCES ===
- poem_id: 0.14% match (6907 differences)
- poet_name_with_diacritics: 33.21% match (4620 differences)

=== FINAL VERDICT ===
✅ SUCCESS: The two DataFrames are substantially identical!
💡 Key findings:
   • All 6,917 poems from DF2 are present in DF1
   • 26 out of 28 common columns have identical values
   • Only differences are technical (IDs and diacritics)
   • DF1 contains 67 additional poems not i